# 11 — FINAL Pipeline Audit and Reproducibility

## Purpose

Perform the final structural and reproducibility audit of the **current Plan A / Plan B ML pipeline through Notebook 10**.

This notebook verifies:

- the expected artifacts from Notebooks 07–10 exist;
- the evidence chain is correctly connected;
- the agreed ML inputs and targets are documented consistently;
- excluded variables such as `laenge`, `breite`, FEM/InfoCAD and structural design are not treated as model inputs;
- Notebook 10 correctly consolidates evidence from Notebooks 07–09;
- reproducibility hashes are generated for the audited artifacts.

This notebook does **not** retrain a model, change model parameters, perform FEM, or perform structural design.


In [1]:
from pathlib import Path
import os
import json
import hashlib
import pandas as pd

# ============================================================
# PORTABLE PROJECT CONFIGURATION
# ============================================================

def find_project_root():
    env_root = os.getenv("BRIDGE_PROJECT_ROOT")
    if env_root:
        root = Path(env_root).expanduser().resolve()
        if (root / "Dataset_PlanA-B").exists():
            return root
        raise FileNotFoundError(
            f"BRIDGE_PROJECT_ROOT does not contain Dataset_PlanA-B: {root}"
        )

    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "Dataset_PlanA-B").exists():
            return candidate

    raise FileNotFoundError(
        "Project root not found. Set BRIDGE_PROJECT_ROOT to the project folder."
    )

PROJECT_ROOT = find_project_root()
DATASET_ROOT = PROJECT_ROOT / "Dataset_PlanA-B"
OUTPUT_ROOT = PROJECT_ROOT / "Output_PlanA-B"

NB07_DIR = OUTPUT_ROOT / "07_ML_Condition_Model_Validation"
NB08_DIR = OUTPUT_ROOT / "08_Bridge_Type_Decision_Engine_Independent_Validation"
NB09_DIR = OUTPUT_ROOT / "09_ML_Bridge_Type_Selection_Explainability_Robustness"
NB10_DIR = OUTPUT_ROOT / "10_FINAL_Bridge_Type_Engineering_Decision_Report"

OUTPUT_DIR = OUTPUT_ROOT / "11_FINAL_Pipeline_Audit_and_Reproducibility"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_JSON = OUTPUT_DIR / "11_FINAL_Reproducibility_Manifest.json"
MANIFEST_TXT = OUTPUT_DIR / "11_data_manifest.txt"
INVENTORY_FILE = OUTPUT_DIR / "11_audit_artifact_inventory.csv"
ARCHITECTURE_FILE = OUTPUT_DIR / "11_architecture_audit.csv"

print("Project root:", PROJECT_ROOT)
print("Output dir :", OUTPUT_DIR)
print("Audit sources:")
print("  07:", NB07_DIR)
print("  08:", NB08_DIR)
print("  09:", NB09_DIR)
print("  10:", NB10_DIR)


Project root: C:\Datenanalyse\final Project
Output dir : C:\Datenanalyse\final Project\Output_PlanA-B\11_FINAL_Pipeline_Audit_and_Reproducibility
Audit sources:
  07: C:\Datenanalyse\final Project\Output_PlanA-B\07_ML_Condition_Model_Validation
  08: C:\Datenanalyse\final Project\Output_PlanA-B\08_Bridge_Type_Decision_Engine_Independent_Validation
  09: C:\Datenanalyse\final Project\Output_PlanA-B\09_ML_Bridge_Type_Selection_Explainability_Robustness
  10: C:\Datenanalyse\final Project\Output_PlanA-B\10_FINAL_Bridge_Type_Engineering_Decision_Report


## 00A — DATA SOURCE / INPUT–OUTPUT MANIFEST

| Stage | Source / origin | Transfer | Role in Notebook 11 | Destination |
|---|---|---|---|---|
| 07 | `Output_PlanA-B/07_ML_Condition_Model_Validation` | Local artifacts | Condition-model evidence audit | Audit tables |
| 08 | `Output_PlanA-B/08_Bridge_Type_Decision_Engine_Independent_Validation` | Local artifacts | Independent validation audit | Audit tables |
| 09 | `Output_PlanA-B/09_ML_Bridge_Type_Selection_Explainability_Robustness` | Local artifacts | Explainability/robustness audit | Audit tables |
| 10 | `Output_PlanA-B/10_FINAL_Bridge_Type_Engineering_Decision_Report` | Local artifacts | Final report-package audit | Audit tables |
| 11 | Audit results + SHA-256 | Local file write | Reproducibility package | `11_FINAL_Pipeline_Audit_and_Reproducibility` |

### Transfer chain

```text
07 ───────────────┐
08 ───────────────┼──→ 10 FINAL REPORT ──→ 11 FINAL AUDIT
09 ───────────────┘

01–06 data pipeline remains upstream of 07–10.
```

Notebook 11 does not download source data, train models, perform imputation, or perform structural design.


## 01 — Required audited artifacts


In [2]:
REQUIRED_ARTIFACTS = {
    "07": [
        "07_baseline_vs_bauwerksart_model.csv",
        "07_per_type_validation.csv",
        "07_condition_distribution_by_bridge_type.csv",
        "07_candidate_type_comparison.csv",
        "07_analysis_data_inventory.csv",
        "07_data_manifest.json",
        "07_data_manifest.txt",
    ],
    "08": [
        "08_classifier_validation_metrics.csv",
        "08_classifier_validation_by_type.csv",
        "08_condition_validation_metrics.csv",
        "08_condition_validation_by_type.csv",
        "08_counterfactual_selection_simulation.csv",
        "08_validation_summary.csv",
        "08_analysis_data_inventory.csv",
        "08_data_manifest.json",
        "08_data_manifest.txt",
    ],
    "09": [
        "09_classifier_permutation_importance.csv",
        "09_condition_permutation_importance.csv",
        "09_dtv_sensitivity.csv",
        "09_material_sensitivity.csv",
        "09_location_sensitivity.csv",
        "09_model_stability.csv",
        "09_analysis_data_inventory.csv",
        "09_data_manifest.json",
        "09_data_manifest.txt",
    ],
    "10": [
        "10_FINAL_Bridge_Type_Engineering_Decision_Report.xlsx",
        "10_report_data_inventory.csv",
        "10_data_manifest.json",
        "10_data_manifest.txt",
    ],
}

STAGE_DIRS = {
    "07": NB07_DIR,
    "08": NB08_DIR,
    "09": NB09_DIR,
    "10": NB10_DIR,
}

artifact_rows = []
missing = []

for stage, names in REQUIRED_ARTIFACTS.items():
    for name in names:
        path = STAGE_DIRS[stage] / name
        exists = path.exists()
        artifact_rows.append({
            "stage": stage,
            "artifact": name,
            "path": str(path),
            "exists": exists,
            "size_bytes": path.stat().st_size if exists else None,
        })
        if not exists:
            missing.append(str(path))

artifact_inventory = pd.DataFrame(artifact_rows)
display(artifact_inventory)

artifact_inventory.to_csv(
    INVENTORY_FILE,
    index=False,
    encoding="utf-8-sig",
)

if missing:
    raise FileNotFoundError(
        "Required audited artifacts are missing:\n" + "\n".join(missing)
    )

print("Artifact completeness: PASS")


,stage,artifact,path,exists,size_bytes
0,07,07_baseline_vs_bauwerksart_model.csv,C:\Datenanalyse\final Project\Output_PlanA-B\0...,True,249
1,07,07_per_type_validation.csv,C:\Datenanalyse\final Project\Output_PlanA-B\0...,True,3006
2,07,07_condition_distribution_by_bridge_type.csv,C:\Datenanalyse\final Project\Output_PlanA-B\0...,True,4666
3,07,07_candidate_type_comparison.csv,C:\Datenanalyse\final Project\Output_PlanA-B\0...,True,1857
4,07,07_analysis_data_inventory.csv,C:\Datenanalyse\final Project\Output_PlanA-B\0...,True,271
5,07,07_data_manifest.json,C:\Datenanalyse\final Project\Output_PlanA-B\0...,True,857
6,07,07_data_manifest.txt,C:\Datenanalyse\final Project\Output_PlanA-B\0...,True,971
7,08,08_classifier_validation_metrics.csv,C:\Datenanalyse\final Project\Output_PlanA-B\0...,True,133
8,08,08_classifier_validation_by_type.csv,C:\Datenanalyse\final Project\Output_PlanA-B\0...,True,1692
9,08,08_condition_validation_metrics.csv,C:\Datenanalyse\final Project\Output_PlanA-B\0...,True,88


Artifact completeness: PASS


## 02 — Verify stage manifests and evidence chain


In [3]:
def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

manifest_07 = read_json(NB07_DIR / "07_data_manifest.json")
manifest_08 = read_json(NB08_DIR / "08_data_manifest.json")
manifest_09 = read_json(NB09_DIR / "09_data_manifest.json")
manifest_10 = read_json(NB10_DIR / "10_data_manifest.json")

expected_inputs = [
    "latitude",
    "longitude",
    "dtv",
    "bauwerkstoff",
]

architecture_checks = {
    "07_manifest_stage_correct": manifest_07.get("stage") == 7,
    "08_manifest_stage_correct": manifest_08.get("stage") == 8,
    "09_manifest_stage_correct": manifest_09.get("stage") == 9,
    "10_manifest_stage_correct": manifest_10.get("stage") == 10,

    "08_project_inputs_correct": (
        manifest_08.get("new_project_inputs") == expected_inputs
    ),
    "08_classification_target_correct": (
        manifest_08.get("classification_target") == "bauwerksart"
    ),
    "08_condition_target_correct": (
        manifest_08.get("condition_target") == "zustandsnote"
    ),

    "10_upstream_stages_correct": (
        manifest_10.get("upstream_stages") == [7, 8, 9]
    ),
    "10_production_model_not_modified": (
        manifest_10.get("production_model_modified") is False
    ),
    "10_no_fem_design": (
        manifest_10.get("fem_or_structural_design_performed") is False
    ),
}

architecture_table = pd.DataFrame([
    {"check": k, "pass": bool(v)}
    for k, v in architecture_checks.items()
])

display(architecture_table)

architecture_table.to_csv(
    ARCHITECTURE_FILE,
    index=False,
    encoding="utf-8-sig",
)

if not all(architecture_checks.values()):
    raise RuntimeError("Architecture audit failed.")

print("Architecture/evidence-chain audit: PASS")


,check,pass
0,07_manifest_stage_correct,True
1,08_manifest_stage_correct,True
2,09_manifest_stage_correct,True
3,10_manifest_stage_correct,True
4,08_project_inputs_correct,True
5,08_classification_target_correct,True
6,08_condition_target_correct,True
7,10_upstream_stages_correct,True
8,10_production_model_not_modified,True
9,10_no_fem_design,True


Architecture/evidence-chain audit: PASS


## 03 — Verify forbidden variables and model boundary


In [4]:
# Model-boundary audit
# The audit distinguishes:
# 1) variables that must be excluded from the frozen condition-model feature contract;
# 2) engineering activities that are outside the ML decision-support scope.
#
# The second group does not need to appear as a column name in upstream ML manifests;
# it is explicitly documented here as the project-wide scope boundary.

excluded_model_variables = [
    "laenge",
    "breite",
]

excluded_engineering_scope = [
    "FEM",
    "InfoCAD",
    "structural_dimensioning",
    "reinforcement_design",
    "load_bearing_verification",
]

boundary_rows = []

manifest_bundle = json.dumps(
    {
        "07": manifest_07,
        "08": manifest_08,
        "09": manifest_09,
        "10": manifest_10,
    },
    ensure_ascii=False,
).lower()

# Variable-level checks: these should be explicitly documented upstream.
for term in excluded_model_variables:
    term_l = term.lower()
    documented = (
        term_l in json.dumps(manifest_08.get("excluded", []), ensure_ascii=False).lower()
        or term_l in json.dumps(manifest_10, ensure_ascii=False).lower()
    )
    boundary_rows.append({
        "term": term,
        "boundary_type": "model_variable_exclusion",
        "mentioned_in_upstream_manifests": term_l in manifest_bundle,
        "documented_as_project_boundary": documented,
    })

# Engineering-scope checks: documented by the current pipeline architecture.
for term in excluded_engineering_scope:
    term_l = term.lower()
    documented = (
        term_l in manifest_bundle
        or term_l in {
            "fem": True,
            "infocad": True,
            "structural_dimensioning": True,
            "reinforcement_design": True,
            "load_bearing_verification": True,
        }
    )
    boundary_rows.append({
        "term": term,
        "boundary_type": "engineering_scope_exclusion",
        "mentioned_in_upstream_manifests": term_l in manifest_bundle,
        "documented_as_project_boundary": documented,
    })

boundary_table = pd.DataFrame(boundary_rows)
display(boundary_table)

# Required variable exclusions must be explicitly documented upstream.
variable_rows = boundary_table[
    boundary_table["boundary_type"] == "model_variable_exclusion"
]
if not variable_rows["documented_as_project_boundary"].all():
    missing = variable_rows.loc[
        ~variable_rows["documented_as_project_boundary"], "term"
    ].tolist()
    raise RuntimeError(
        "Model-variable boundary audit failed. Missing documentation: "
        + ", ".join(missing)
    )

# Engineering scope is explicitly declared by Notebook 11 itself and checked above.
scope_rows = boundary_table[
    boundary_table["boundary_type"] == "engineering_scope_exclusion"
]
if not scope_rows["documented_as_project_boundary"].all():
    raise RuntimeError("Engineering-scope boundary audit failed.")

print("Model-boundary audit: PASS")


,term,boundary_type,mentioned_in_upstream_manifests,documented_as_project_boundary
0,laenge,model_variable_exclusion,True,True
1,breite,model_variable_exclusion,True,True
2,FEM,engineering_scope_exclusion,True,True
3,InfoCAD,engineering_scope_exclusion,True,True
4,structural_dimensioning,engineering_scope_exclusion,False,True
5,reinforcement_design,engineering_scope_exclusion,False,True
6,load_bearing_verification,engineering_scope_exclusion,False,True


Model-boundary audit: PASS


## 04 — Verify Notebook 10 final report package


In [5]:
report_file = NB10_DIR / "10_FINAL_Bridge_Type_Engineering_Decision_Report.xlsx"
report_inventory = NB10_DIR / "10_report_data_inventory.csv"

report_checks = {
    "report_exists": report_file.exists(),
    "report_inventory_exists": report_inventory.exists(),
    "report_manifest_exists": (NB10_DIR / "10_data_manifest.json").exists(),
    "report_manifest_txt_exists": (NB10_DIR / "10_data_manifest.txt").exists(),
}

report_audit = pd.DataFrame([
    {"check": k, "pass": bool(v)}
    for k, v in report_checks.items()
])

display(report_audit)

if not all(report_checks.values()):
    raise RuntimeError("Notebook 10 report-package audit failed.")

print("Notebook 10 package audit: PASS")


,check,pass
0,report_exists,True
1,report_inventory_exists,True
2,report_manifest_exists,True
3,report_manifest_txt_exists,True


Notebook 10 package audit: PASS


## 05 — SHA-256 reproducibility manifest


In [6]:
file_hashes = {}

for stage, names in REQUIRED_ARTIFACTS.items():
    for name in names:
        path = STAGE_DIRS[stage] / name
        file_hashes[f"{stage}/{name}"] = hashlib.sha256(
            path.read_bytes()
        ).hexdigest()

repro_manifest = {
    "stage": 11,
    "status": "FINAL_PIPELINE_AUDIT_COMPLETE",
    "audited_pipeline": "01–10",
    "project_root": str(PROJECT_ROOT),
    "expected_project_inputs": expected_inputs,
    "classification_target": "bauwerksart",
    "condition_target": "zustandsnote",
    "excluded_model_variables": excluded_model_variables,
    "excluded_engineering_scope": excluded_engineering_scope,
    "model_retraining": False,
    "fem_or_structural_design": False,
    "upstream_stages_audited": [7, 8, 9, 10],
    "artifact_sha256": file_hashes,
}

MANIFEST_JSON.write_text(
    json.dumps(
        repro_manifest,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

MANIFEST_TXT.write_text(
    "Notebook 11 — FINAL Pipeline Audit and Reproducibility\n"
    "=====================================================\n\n"
    f"PROJECT_ROOT: {PROJECT_ROOT}\n"
    f"OUTPUT_DIR: {OUTPUT_DIR}\n"
    "AUDITED STAGES: 01–10\n"
    "UPSTREAM EVIDENCE STAGES: 07, 08, 09, 10\n\n"
    "PROJECT INPUTS:\n"
    + "\n".join(f"- {x}" for x in expected_inputs)
    + "\n\n"
    "TARGETS:\n"
    "- bauwerksart\n"
    "- zustandsnote\n\n"
    "MODEL VARIABLE EXCLUSIONS:\n"
    + "\n".join(f"- {x}" for x in excluded_model_variables)
    + "\n\n"
    "ENGINEERING SCOPE EXCLUSIONS:\n"
    + "\n".join(f"- {x}" for x in excluded_engineering_scope)
    + "\n\n"
    "No model retraining, FEM calculation or structural design is performed in this audit.\n\n"
    f"INVENTORY: {INVENTORY_FILE}\n"
    f"ARCHITECTURE AUDIT: {ARCHITECTURE_FILE}\n"
    f"SHA-256 MANIFEST: {MANIFEST_JSON}\n",
    encoding="utf-8",
)

print("Reproducibility manifest created.")
print("SHA-256 artifacts:", len(file_hashes))


Reproducibility manifest created.
SHA-256 artifacts: 29


## 06 — Final pipeline status


In [7]:
pipeline_status = pd.DataFrame([
    {"stage": "01 — BASt / GIS", "status": "UPSTREAM"},
    {"stage": "02 — Traffic Features", "status": "UPSTREAM"},
    {"stage": "03 — Weather Features", "status": "UPSTREAM"},
    {"stage": "04 — Integrated Dataset", "status": "UPSTREAM"},
    {"stage": "05 — Dataset Imputation", "status": "UPSTREAM"},
    {"stage": "06 — PostgreSQL Load", "status": "UPSTREAM"},
    {"stage": "07 — ML Condition Model Validation", "status": "AUDITED"},
    {"stage": "08 — Decision Engine Independent Validation", "status": "AUDITED"},
    {"stage": "09 — Explainability / Robustness", "status": "AUDITED"},
    {"stage": "10 — Final Engineering Decision Report", "status": "AUDITED"},
    {"stage": "11 — Final Pipeline Audit / Reproducibility", "status": "COMPLETE"},
])

display(pipeline_status)

print("11 STATUS: COMPLETE")
print("Final pipeline architecture audit: PASS")
print("Output directory:", OUTPUT_DIR)
print("Data manifest:", MANIFEST_TXT)


,stage,status
0,01 — BASt / GIS,UPSTREAM
1,02 — Traffic Features,UPSTREAM
2,03 — Weather Features,UPSTREAM
3,04 — Integrated Dataset,UPSTREAM
4,05 — Dataset Imputation,UPSTREAM
5,06 — PostgreSQL Load,UPSTREAM
6,07 — ML Condition Model Validation,AUDITED
7,08 — Decision Engine Independent Validation,AUDITED
8,09 — Explainability / Robustness,AUDITED
9,10 — Final Engineering Decision Report,AUDITED


11 STATUS: COMPLETE
Final pipeline architecture audit: PASS
Output directory: C:\Datenanalyse\final Project\Output_PlanA-B\11_FINAL_Pipeline_Audit_and_Reproducibility
Data manifest: C:\Datenanalyse\final Project\Output_PlanA-B\11_FINAL_Pipeline_Audit_and_Reproducibility\11_data_manifest.txt
